<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Caracterización de los productos con oportunidad de exportación

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

Define el umbral de proximidad en el espacio de producto y el umbral de valor comercial.

In [3]:
cercania = 0.65

valor_minimo = 50   # miles de dólares

### Lectura de datos

Carga las bases preparadas en la cadena de limpieza: la correspondencia arancelaria, la
clasificación de Leamer, el comercio bilateral y la matriz de proximidad entre productos.

**Pasos realizados:**

1. **Comercio con clasificación sectorial:** incorpora la categoría de Leamer a la base de comercio bilateral.

2. **Canasta exportadora actual:** identifica los productos que Colombia comercia por encima del umbral de valor y los marca como oportunidades a profundizar.

3. **Espacio de producto acotado:** restringe la matriz de proximidad a los productos de esa canasta.

In [4]:
df_hs2012_2017 = pd.read_parquet(temp / 'df_hs2012_2017.parquet')
df_leamer = pd.read_parquet(temp / 'df_leamer.parquet')

df_comercio = pd.read_parquet(temp / 'df_comercio.parquet')
df_comercio = df_comercio.merge(df_leamer[['HS6', 'Leamer']], on='HS6', how='left')

df_comercio_col = pd.read_parquet(temp / 'df_comercio_mundial.parquet').query("PartnerISO3 == 'COL' and `TradeValue in 1000 USD` >= @valor_minimo")[['ProductCode']].rename(columns={'ProductCode': 'HS6'}).drop_duplicates()
df_comercio_col['HS6'] = df_comercio_col['HS6'].astype(str).str.zfill(6)
df_comercio_col['Margen'] = 'Oportunidades a profundizar'
list_comercio_col = df_comercio_col['HS6'].tolist()

df_product_space = pd.read_parquet(temp / 'df_product_space.parquet')

df_product_space_col = df_product_space[list_comercio_col].copy()

### Productos priorizados por fuente

Consolida las listas de productos recibidas de terceros.

**Fuentes y categorías:**

1. **Banco Interamericano de Desarrollo:** cuatro hojas del archivo de identificación de productos, correspondientes a oportunidades de corto plazo, mediano plazo, intrarregionales y de largo plazo. Estas últimas llegan a cuatro dígitos y se expanden a seis mediante la base de comercio.

2. **Araújo Ibarra:** los sesenta y cinco productos principales de los cinco mayores exportadores a Estados Unidos, expresados en HS 2017 y traducidos a HS 2012.

3. **AmCham:** productos de interés en el mercado estadounidense.

In [5]:
dfs_BID = pd.read_excel(raw / 'insumos' / 'Col - ID Productos.xlsx', sheet_name=None)

df_qw = dfs_BID['QW'][['HS 6 CODE']].drop_duplicates().rename(columns={'HS 6 CODE': 'HS6'})
df_qw['HS6'] = df_qw['HS6'].astype(str).str.zfill(6)
df_qw['Fuente'] = 'BID'
df_qw['Categoría'] = 'Quick Win'
print(f'''Ingresaron {df_qw['HS6'].nunique()} productos de corto plazo''')

df_mt = dfs_BID['MT'][['HS 6 CODE']].drop_duplicates().rename(columns={'HS 6 CODE': 'HS6'})
df_mt['HS6'] = df_mt['HS6'].astype(str).str.zfill(6)
df_mt['Fuente'] = 'BID'
df_mt['Categoría'] = 'Mid Term'
print(f'''Ingresaron {df_mt['HS6'].nunique()} productos de mediano plazo''')

df_lac = dfs_BID['INTRA LAC'][['HS 6 CODE']].drop_duplicates().rename(columns={'HS 6 CODE': 'HS6'})
df_lac['HS6'] = df_lac['HS6'].astype(str).str.zfill(6)
df_lac['Fuente'] = 'BID'
df_lac['Categoría'] = 'Intra LAC'
print(f'''Ingresaron {df_lac['HS6'].nunique()} productos de LAC''')

df_lt = dfs_BID['LT'][['HS 4 CODE']].drop_duplicates().rename(columns={'HS 4 CODE': 'HS4'})
print(f'''Ingresaron {df_lt['HS4'].nunique()} productos de largo plazo''')
df_lt['HS4'] = df_lt['HS4'].astype(str).str.zfill(4)
df_lt['Fuente'] = 'BID'
df_lt['Categoría'] = 'Long Term'
df_lt = df_lt.merge(df_comercio[['HS4','HS6']].drop_duplicates(),how='inner',on='HS4')[['HS6','Fuente','Categoría']].drop_duplicates()
print(f'''Se expandieron a {df_lt['HS6'].nunique()} productos de largo plazo''')

df_AI = pd.read_excel(raw / 'insumos' / 'Top 65 productos de los top 5 exportadores.xlsx', sheet_name='TOP 65 TOP 5').dropna()[['Código producto']]

df_AI['HS6_17'] = df_AI['Código producto'].astype(str).str[:6].str.zfill(6)
print(f'''Ingresaron {df_AI['HS6_17'].nunique()} productos de Araújo Ibarra''')
df_AI['Fuente'] = 'Araújo Ibarra'
df_AI['Categoría'] = 'Top 65 productos de los top 5 exportadores a USA'
df_AI = df_AI.merge(df_hs2012_2017, on='HS6_17', how='left')
df_AI = df_AI[['HS6','Fuente','Categoría']].drop_duplicates()
print(f'''Se contrajeron a {df_AI['HS6'].nunique()} productos de Araújo Ibarra''')

df_amcham = pd.read_excel(raw / 'insumos' / 'Amcham.xlsx')[['HS6']]

df_amcham['HS6'] = df_amcham['HS6'].astype(str).str.zfill(6)
df_amcham['Fuente'] = 'Foco EEUU'
df_amcham['Categoría'] = 'Productos Foco EEUU'
print(f'''Ingresaron {df_amcham['HS6'].nunique()} productos de AmCham''')

Ingresaron 50 productos de corto plazo
Ingresaron 50 productos de mediano plazo
Ingresaron 50 productos de LAC
Ingresaron 50 productos de largo plazo


Se expandieron a 226 productos de largo plazo
Ingresaron 65 productos de Araújo Ibarra
Se contrajeron a 64 productos de Araújo Ibarra
Ingresaron 56 productos de AmCham


### Consolidación de la cartera

Concatena las cuatro fuentes e indica cuáles de esos productos exporta Colombia actualmente.

In [6]:
df_products = pd.concat([df_qw, df_mt, df_lac, df_lt, df_AI, df_amcham], ignore_index=True).merge(df_comercio_col, how='left', on='HS6')
print(f'''En total hay {len(df_products)} productos. De los cuales, {len(df_products.dropna())} son productos que Colombia exporta actualmente''')

En total hay 496 productos. De los cuales, 311 son productos que Colombia exporta actualmente


### Redes de cercanía en el espacio de producto

Para cada producto de la canasta exportadora actual separa los productos cercanos, cuya
proximidad alcanza el umbral, de los lejanos. Descarta las entradas sin relaciones.

In [7]:
dict_productos_cerca = dict()
dict_productos_lejos = dict()
for producto in df_product_space_col:
    dict_productos_cerca[producto] = df_product_space_col[producto].drop(producto).reset_index().query(f"`{producto}`>={cercania}")['index'].to_list()
    dict_productos_lejos[producto] = df_product_space_col[producto].drop(producto).reset_index().query(f"`{producto}`<{cercania}")['index'].to_list()
dict_productos_cerca = {k: v for k, v in dict_productos_cerca.items() if v}
dict_productos_lejos = {k: v for k, v in dict_productos_lejos.items() if v}

### Margen de oportunidad de los productos no exportados

Asigna el margen a los productos que no forman parte de la canasta actual: oportunidad a la
vista si son cercanos a algún producto exportado, y de largo alcance en caso contrario.

**Nota de auditoría:** la variable `tuple_margen` conserva el valor de la iteración anterior
cuando un producto no cae en ninguna de las dos condiciones. El código se migra tal cual.

In [8]:
list_margen = list()

for productos in df_products[df_products.isnull().any(axis=1)]['HS6'].unique().tolist():
    for producto_lejos in dict_productos_lejos.keys():
        if productos in dict_productos_lejos[producto_lejos]:
            tuple_margen = (productos,'Oportunidades de largo alcance')
    for producto_cerca in dict_productos_cerca.keys():
        if productos in dict_productos_cerca[producto_cerca]:
            tuple_margen = (productos,'Oportunidades a la vista')
    list_margen.append(tuple_margen)

df_products = df_products.merge(pd.DataFrame(list_margen,columns=['HS6','Margen']),how='left',on='HS6').drop_duplicates().reset_index(drop=True)
df_products['Margen'] = df_products['Margen_x'].fillna(df_products['Margen_y'])
df_products = df_products.drop(columns=['Margen_x','Margen_y']).dropna().reset_index(drop=True)

### Productos adicionales identificados en el espacio de producto

Amplía la cartera con productos que Colombia aún no exporta pero que son cercanos a alguno de
los ya priorizados, sin depender de listas externas.

**Pasos realizados:**

1. **Búsqueda de vecinos:** recupera los productos cuya proximidad con algún producto priorizado alcanza el umbral, y descarta los que ya están en la cartera.

2. **Etiquetado:** marca los productos hallados como identificados por Fedesarrollo mediante el espacio de producto.

3. **Asignación de margen:** aplica el mismo criterio de cercanía y conserva únicamente las oportunidades a la vista.

In [9]:
list_productos_nuevos_cerca = list()
for productos in df_products['HS6'].unique().tolist():
    try:
        list_productos_nuevos_cerca += df_product_space[[productos]].query(f"`{productos}`>={cercania}").drop(productos).reset_index()['index'].to_list()
    except:
        pass
list_productos_nuevos_cerca = list({x for x in set(list_productos_nuevos_cerca) if x not in df_products['HS6'].unique().tolist()})

df_Fed = pd.DataFrame(list_productos_nuevos_cerca, columns=['HS6']).drop_duplicates()
df_Fed['Fuente'] = 'Espacio Producto'
df_Fed['Categoría'] = 'Fedesarrollo'
df_Fed = df_Fed.merge(df_comercio_col, how='left', on='HS6')

list_margen_fed = list()
for productos in df_Fed[df_Fed.isnull().any(axis=1)]['HS6'].unique().tolist():
    for producto_lejos in dict_productos_lejos.keys():
        if productos in dict_productos_lejos[producto_lejos]:
            tuple_margen = (productos,'Oportunidades de largo alcance')
    for producto_cerca in dict_productos_cerca.keys():
        if productos in dict_productos_cerca[producto_cerca]:
            tuple_margen = (productos,'Oportunidades a la vista')
    list_margen_fed.append(tuple_margen)

df_Fed = df_Fed.merge(pd.DataFrame(list_margen_fed,columns=['HS6','Margen']),how='left',on='HS6').drop_duplicates().reset_index(drop=True)
df_Fed['Margen'] = df_Fed['Margen_x'].fillna(df_Fed['Margen_y'])
df_Fed.drop(columns=['Margen_x','Margen_y'], inplace=True)

df_Fed = df_Fed.query("Margen != 'Oportunidades de largo alcance'")
df_products = pd.concat([df_products, df_Fed], ignore_index=True)

print(f'''Los productos encontrados por Fedesarrollo son {df_Fed['HS6'].nunique()}''')

Los productos encontrados por Fedesarrollo son 232


### Clasificación sectorial y por uso económico

Incorpora la categoría de Leamer y la clasificación por grandes categorías económicas, que
distingue entre formación bruta de capital fijo, consumo intermedio y consumo final.

In [10]:
df_products = df_products.merge(df_leamer, how='left')
df_products = df_products.merge(pd.read_parquet(temp / 'df_BEC.parquet'), how='left', on='HS6')

temp.mkdir(parents=True, exist_ok=True)
df_products.to_parquet(temp / 'df_products_caracterizado.parquet', index=False)
print(f"productos caracterizados: {df_products['HS6'].nunique()}")

productos caracterizados: 662
